# Introduction to Python Threading

Python threading allows you to have different parts of your program run concurrently and can simplify your design.

See:
- https://realpython.com/intro-to-python-threading/


## threading.Thread()

Create and start a thread

In [2]:
# Simple threading

import logging
import threading
import time

def some_function(name):
    logging.info("Thread %s: starting", name)
    time.sleep(2)
    logging.info("Thread %s: finishing", name)

format = "%(asctime)s: %(message)s"
logging.basicConfig(format=format, level=logging.INFO, datefmt="%H:%M:%S")

# Create a thread and assign some_function and passing arguments to it
t = threading.Thread(target=some_function, args=(1,))
logging.info("Main    : before running thread")
t.start()

# t.join: Wait for the thread to finish
logging.info("Main    : wait for the thread to finish")
t.join()
logging.info("Main    : all done")

18:30:42: Main    : before running thread
18:30:42: Thread 1: starting
18:30:42: Main    : wait for the thread to finish
18:30:44: Thread 1: finishing
18:30:44: Main    : all done


## threading.Thread.join()
Working with multiple threads

In [9]:
# Working with multiple threads

threads = list()
for index in range(3):
    logging.info("Main    : create and start thread %d.", index)
    t = threading.Thread(target=thread_function, args=(index,))
    threads.append(t)
    t.start()
for index, thread in enumerate(threads):
    logging.info("Main    : joining thread %d.", index)
    thread.join()
    logging.info("Main    : done joining thread %d", index)

18:00:39: Main    : create and start thread 0.
18:00:39: Thread 0: starting
18:00:39: Main    : create and start thread 1.
18:00:39: Thread 1: starting
18:00:39: Main    : create and start thread 2.
18:00:39: Thread 2: starting
18:00:39: Main    : joining thread 0.
18:00:41: Thread 0: finishing
18:00:41: Main    : done joining thread 0
18:00:41: Main    : joining thread 1.
18:00:41: Thread 1: finishing
18:00:41: Main    : done joining thread 1
18:00:41: Main    : joining thread 2.
18:00:41: Thread 2: finishing
18:00:41: Main    : done joining thread 2


## concurrent.futures.ThreadPoolExecutor

In the following example, the code creates a ThreadPoolExecutor as a context manager, telling it how many worker threads it wants in the pool. It then uses .map() to step through an iterable of things, in your case range(3), passing each one to a thread in the pool.

The end of the with block causes the ThreadPoolExecutor to do a .join() on each of the threads in the pool. It is strongly recommended that you use ThreadPoolExecutor as a context manager when you can so that you never forget to .join() the threads.

In [11]:
# Using a ThreadPoolExecutor
import concurrent.futures
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    executor.map(some_function, range(3))

18:07:11: Thread 0: starting
18:07:11: Thread 1: starting
18:07:11: Thread 2: starting
18:07:13: Thread 1: finishing
18:07:13: Thread 0: finishing
18:07:13: Thread 2: finishing


## Race Conditions

Race conditions can occur when two or more threads access a shared piece of data or resource. In this example, you’re going to create a large race condition that happens every time, but be aware that most race conditions are not this obvious. Frequently, they only occur rarely, and they can produce confusing results. As you can imagine, this makes them quite difficult to debug.

In [11]:
class FakeDatabaseWithRaceCondition:
    def __init__(self):
        self.value = 0

    def update(self, name):
        logging.info("Thread %s: starting update", name)
        local_copy = self.value
        local_copy += 1
        time.sleep(0.1)         # simulate a posible race condition as other threads could change value of local_copy
        self.value = local_copy # updated self.value which is a shared variable
        logging.info("Thread %s: finishing update", name)

database = FakeDatabaseWithRaceCondition()
num_thread = 4
logging.info("Testing update. Starting value is %d.", database.value)

import concurrent.futures
with concurrent.futures.ThreadPoolExecutor(max_workers=num_thread) as executor:
    for index in range(num_thread):
        executor.submit(database.update, index)
logging.info("Testing update. Ending value is %d.", database.value)  # Expectation: Ending value is 4.



18:37:37: Testing update. Starting value is 0.
18:37:37: Thread 0: starting update
18:37:37: Thread 1: starting update
18:37:37: Thread 2: starting update
18:37:37: Thread 3: starting update
18:37:37: Thread 0: finishing update
18:37:37: Thread 1: finishing update
18:37:37: Thread 2: finishing update
18:37:37: Thread 3: finishing update
18:37:37: Testing update. Ending value is 1.


## Basic Synchronization Using Lock

To avoid a race condition, you need to find a way to allow only one thread at a time into the read-modify-write section of your code. The most common way to do this is called Lock in Python. In some other languages this same idea is called a mutex. Mutex comes from MUTual EXclusion, which is exactly what a Lock does.

A Lock is an object that acts like a hall pass. Only one thread at a time can have the Lock. Any other thread that wants the Lock must wait until the owner of the Lock gives it up.

The basic functions to do this are .acquire() and .release(). A thread will call my_lock.acquire() to get the lock. If the lock is already held, the calling thread will wait until it is released. There’s an important point here. If one thread gets the lock but never gives it back, your program will be stuck.

Python’s Lock will also operate as a context manager so you can use it in a ***'with'*** statement, and it gets released automatically when the with block exits for any reason.

In [10]:
class FakeDatabase:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def update(self, name):
        logging.info("Thread %s: starting update", name)
        logging.info("Thread %s: about to lock", name)
        with self._lock:  # obtain the lock and will release via end of with statement
            logging.info("Thread %s: has lock", name)
            local_copy = self.value
            local_copy += 1
            time.sleep(0.1)         # simulate a posible race condition as other threads could change value of local_copy
            self.value = local_copy # updated self.value which is a shared variable
            logging.info("Thread %s: current value is %d.", name, self.value)
            logging.info("Thread %s: about to release lock", name)
        logging.info("Thread %s: after release", name)
        logging.info("Thread %s: finishing update", name)
            
database = FakeDatabase()
num_thread = 4
logging.info("Testing update. Starting value is %d.", database.value)
with concurrent.futures.ThreadPoolExecutor(max_workers=num_thread) as executor:
    for index in range(num_thread):
        executor.submit(database.update, index)
logging.info("Testing update. Ending value is %d.", database.value)


18:37:21: Testing update. Starting value is 0.
18:37:21: Thread 0: starting update
18:37:21: Thread 1: starting update
18:37:21: Thread 2: starting update
18:37:21: Thread 2: about to lock
18:37:21: Thread 0: about to lock
18:37:21: Thread 1: about to lock
18:37:21: Thread 3: starting update
18:37:21: Thread 2: has lock
18:37:21: Thread 3: about to lock
18:37:21: Thread 2: current value is 1.
18:37:21: Thread 2: about to release lock
18:37:21: Thread 2: after release
18:37:21: Thread 2: finishing update
18:37:21: Thread 0: has lock
18:37:21: Thread 0: current value is 2.
18:37:21: Thread 0: about to release lock
18:37:21: Thread 0: after release
18:37:21: Thread 1: has lock
18:37:21: Thread 0: finishing update
18:37:21: Thread 1: current value is 3.
18:37:21: Thread 1: about to release lock
18:37:22: Thread 1: after release
18:37:22: Thread 1: finishing update
18:37:22: Thread 3: has lock
18:37:22: Thread 3: current value is 4.
18:37:22: Thread 3: about to release lock
18:37:22: Thread

## Deadlock

A deadlock occurs when two or more threads are blocked indefinitely, each waiting for a resource held by the other. The most common cause is acquiring multiple locks in a conflicting order.

In [12]:
import threading
import time

# Two shared resources (locks)
lock_a = threading.Lock()
lock_b = threading.Lock()

def thread_one_task():
    print("Thread 1: Attempting to acquire Lock A...")
    with lock_a:
        print("Thread 1: Acquired Lock A. Sleeping...")
        time.sleep(1)  # Ensure Thread 2 has time to grab Lock B
        print("Thread 1: Attempting to acquire Lock B...")
        with lock_b:
            print("Thread 1: Acquired Lock B!")

def thread_two_task():
    print("Thread 2: Attempting to acquire Lock B...")
    with lock_b:
        print("Thread 2: Acquired Lock B. Sleeping...")
        time.sleep(1)  # Ensure Thread 1 has time to grab Lock A
        print("Thread 2: Attempting to acquire Lock A...")
        with lock_a:
            print("Thread 2: Acquired Lock A!")

# Creating and starting threads
t1 = threading.Thread(target=thread_one_task)
t2 = threading.Thread(target=thread_two_task)

t1.start()
t2.start()

t1.join()
t2.join()
print("This line will never be reached.")

Thread 1: Attempting to acquire Lock A...
Thread 1: Acquired Lock A. Sleeping...
Thread 2: Attempting to acquire Lock B...
Thread 2: Acquired Lock B. Sleeping...
Thread 1: Attempting to acquire Lock B...
Thread 2: Attempting to acquire Lock A...


KeyboardInterrupt: 

## threading.Event

In Python, a threading.Event is a simple synchronization primitive used for communication between threads. It functions like a thread-safe boolean flag that allows one thread to signal a condition, while other threads wait for that condition to become true.

Core Methods: The threading.Event class provides four primary methods to manage its internal state
- set(): Sets the internal flag to True. Any threads waiting for the event are immediately unblocked.
- wait(timeout=None): Blocks the calling thread until the internal flag is True. If the flag is already true, it returns immediately. You can optionally provide a timeout in seconds.
- clear(): Resets the internal flag to False. Subsequent calls to wait() will block until set() is called again.
- is_set(): Returns True if the internal flag is currently set, and False otherwise.

In [13]:
import threading
import time

# 1. Initialize the event (flag is False by default)
event = threading.Event()

def worker():
    print("Worker: Waiting for the signal...")
    event.wait()  # Blocks here until event.set() is called
    print("Worker: Signal received! Starting task...")

# Start the thread
t = threading.Thread(target=worker)
t.start()

# Simulate some preparation work
time.sleep(2)

print("Main: Sending signal now.")
# 2. Trigger the event
event.set() 
print("Main: End of program") # Note that the worker will continue to run after this statement

Worker: Waiting for the signal...
Main: Sending signal now.
Main: End of program
Worker: Signal received! Starting task...


## threading.Semaphore

A semaphore in Python's threading module is a synchronization primitive that uses an internal counter to manage access to a resource with limited capacity. While a standard lock allows only one thread at a time, a semaphore allows a specific number of threads to enter a critical section simultaneously.

Core Logic
- Counter Initialization: You initialize the semaphore with a fixed value (e.g., 3), representing the maximum allowed concurrent
- threads.Acquire: When a thread calls .acquire(), the counter decrements. If the counter is zero, the thread blocks until another thread finishes.
- Release: When a thread calls .release(), the counter increments, potentially waking up a waiting thread.

In [14]:
# The best practice is to use the semaphore as a context manager (using the with statement) to ensure the resource is released even if an error occurs.

import threading
import time
import random

# Limit access to 3 concurrent threads
max_connections = 3
connection_limit = threading.Semaphore(max_connections)

def access_resource(thread_id):
    print(f"Thread-{thread_id} is waiting for a slot...")
    
    # Use 'with' to automatically acquire and release
    with connection_limit:
        print(f"Thread-{thread_id} entered. Slots occupied.")
        # Simulate work
        time.sleep(random.uniform(1, 3))
        print(f"Thread-{thread_id} is leaving.")

# Create and start 7 threads (only 3 can run at once)
threads = []
for i in range(7):
    t = threading.Thread(target=access_resource, args=(i,))
    threads.append(t)
    t.start()

# Wait for all to finish
for t in threads:
    t.join()

print("All tasks finished.")

Thread-0 is waiting for a slot...
Thread-0 entered. Slots occupied.
Thread-1 is waiting for a slot...
Thread-1 entered. Slots occupied.
Thread-2 is waiting for a slot...
Thread-2 entered. Slots occupied.
Thread-3 is waiting for a slot...
Thread-4 is waiting for a slot...
Thread-5 is waiting for a slot...
Thread-6 is waiting for a slot...
Thread-1 is leaving.
Thread-3 entered. Slots occupied.
Thread-0 is leaving.
Thread-4 entered. Slots occupied.
Thread-2 is leaving.
Thread-5 entered. Slots occupied.
Thread-3 is leaving.
Thread-6 entered. Slots occupied.
Thread-5 is leaving.
Thread-4 is leaving.
Thread-6 is leaving.
All tasks finished.


## threading.Timer

The threading.Timer class in Python is used to schedule a function to run after a specific delay. It runs on a separate thread, allowing your main program to continue executing without waiting.

Basic Usage: To use a timer, you specify the delay in seconds and the function you want to call. You must explicitly call .start() to begin the countdown.

In [16]:
import threading

def hello():
    print("Timer thread: Hello, World!")

# Create a timer that waits 3 seconds before calling hello()
timer = threading.Timer(3.0, hello)
timer.start()

print("The timer has started. Moving on to other tasks...")

The timer has started. Moving on to other tasks...
Timer thread: Hello, World!


## threading.Barrier

A threading.Barrier is a synchronization primitive used to make a fixed number of threads wait for each other at a specific point in their execution. 
No thread can proceed past the barrier until all participating threads have reached it.

In [17]:
import threading
import time
import random

# 1. Initialize the barrier for 3 threads
# All 3 threads must call .wait() before any can continue
barrier = threading.Barrier(3)

def worker_thread(thread_id):
    print(f"Thread {thread_id}: Doing some preparation...")
    time.sleep(random.uniform(1, 3))  # Simulate varying work times
    
    print(f"Thread {thread_id}: Reached the barrier. Waiting for others...")
    
    # 2. Wait at the barrier
    # This blocks the thread until the 3rd thread arrives
    barrier.wait()
    
    print(f"Thread {thread_id}: Barrier released! Starting the task...")

# 3. Create and start 3 threads
threads = []
for i in range(3):
    t = threading.Thread(target=worker_thread, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("Main: All threads have completed.")

Thread 0: Doing some preparation...
Thread 1: Doing some preparation...
Thread 2: Doing some preparation...
Thread 1: Reached the barrier. Waiting for others...
Thread 2: Reached the barrier. Waiting for others...
Thread 0: Reached the barrier. Waiting for others...
Thread 0: Barrier released! Starting the task...
Thread 1: Barrier released! Starting the task...
Thread 2: Barrier released! Starting the task...
Main: All threads have completed.


## threading.local()

The threading.local() function creates a thread-local storage (TLS) object, which allows you to store data that is isolated and unique to each individual thread. Even if multiple threads share the same threading.local instance, the values they assign to it are not visible to one another.

In [20]:
import threading
import time

# 1. Create a thread-local storage object
thread_local = threading.local()

def worker(value):
    # 2. Assign a value to an attribute on the local object; this attribute is unique to the current thread
    thread_local.user_id = value
    
    time.sleep(1)  # Simulate work
    
    # 3. Access the value later; it remains isolated
    print(f"Thread {threading.current_thread().name}: user_id = {thread_local.user_id}")

# Start two threads with different values
t1 = threading.Thread(target=worker, args=(101,), name="A") # Assign 'A' to the new thread
t2 = threading.Thread(target=worker, args=(202,), name="B") # Assign 'B' to the new thread

t1.start()
t2.start()

t1.join()
t2.join()

Thread B: user_id = 202
Thread A: user_id = 101
